# 05 — Walk-Forward Validation
Tune on a rolling train window, evaluate frozen parameters on the next year.
The IS→OOS decay table is the honest measure of how much of notebook 04's
performance is real vs overfit.

In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

In [ ]:
from lab.backtest import StrategyConfig
from lab.experiments import walk_forward

base = StrategyConfig.from_yaml("../configs/short_put_45dte.yaml")
grid = {
    "leg1_delta.target": [0.10, 0.16, 0.30],
    "take_profit":       [0.25, 0.5, None],
}
wf = walk_forward(base, grid, train_years=4, test_years=1,
                  first_year=2010, last_year=2023)
wf.windows.round(3)

In [ ]:
ax = wf.windows.set_index("test")[["is_sharpe_ratio", "oos_sharpe_ratio"]].plot.bar(
    title="In-sample vs out-of-sample Sharpe by test year")
ax.axhline(0, color="k", lw=0.5);

In [ ]:
# Stitched OOS equity: performance using only parameters chosen on past data
oos = wf.oos_trades.sort_values("exit_date")
oos["cum_pnl"] = oos["realized_pnl"].cumsum()
oos.plot(x="exit_date", y="cum_pnl", title="Walk-forward OOS cumulative P&L");

In [ ]:
print("Chosen parameters per window (stability check):")
wf.windows[["train", "test", "best_params"]]